<a href="https://colab.research.google.com/github/aidenjbrown/Intermediate_Data_Science_CS_28/blob/main/scripts/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. What is your outcome variable(s)? How well does it measure the outcome you are interested? How does it
relate to your expectations?

Our outcome variable is depression score. It measures the outcome we are interested in very well because it is the exact criteria the CDC uses to estimate the prevalance of depression across the population.

2. What are your key explanatory variables?

Our key explanatory variables are...

3. What data cleaning did you have to do?

We had to clean the data by only keeping response for each of the depression screening questions if it was between 0 and 3 so the scores could only reach a max of 27 (which is the max score for depression screening in the NHANES.)

4. How did you wrangle the data?

We wrangled the data by performing an inner merge on all of the 4 raw datasets by SEQN number. We also dropped all of the variables we did not need from the raw data by choosing to only keep the subset that we did need.

5. Are you deciding to exclude any observations? If so, why?

We are deciding to exclude any observations that have NA for depression score since it is our key outcome variable and if a participant does not have that score calculated, that participant is not useful for this data analysis.

6. Did you have to create any new variables from existing variables? If so, how and why?

We did have to create the depression score variable because it was not calculated for us. So if the answer to each question related to depression symptoms had between a 0 and 3 as a response, we recorded added it to the other responses for the same participant for all the depression symptoms to get a score with the max being 27. We wanted to create this variable so we could determine if certain predictors were associated with depression more broadly rather than invidual symptoms only.

In [79]:
# Importing the data

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

depression = pd.read_sas('/content/drive/MyDrive/DataScienceXPTFiles/DPQ_J.xpt')
depression.to_csv('/content/drive/MyDrive/DataScienceXPTFiles/Depression_Scores.csv', index=False)

income = pd.read_sas('/content/drive/MyDrive/DataScienceXPTFiles/INQ_J.xpt')
income.to_csv('/content/drive/MyDrive/DataScienceXPTFiles/Income.csv', index=False)

demographics = pd.read_sas('/content/drive/MyDrive/DataScienceXPTFiles/DEMO_J.xpt')
demographics.to_csv('/content/drive/MyDrive/DataScienceXPTFiles/Demographics.csv', index=False)

sleep = pd.read_sas('/content/drive/MyDrive/DataScienceXPTFiles/SLQ_J.xpt')
sleep.to_csv('/content/drive/MyDrive/DataScienceXPTFiles/Sleep.csv', index=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [80]:
# Now we will merge all of the data based on the SEQN number. Using inner merge to make sure the SEQN exists in both datasets.

merge1 = pd.merge(depression, income, on='SEQN', how = "inner")
print(merge1.shape)
merge2 = pd.merge(merge1, demographics, on='SEQN', how = "inner")
print(merge2.shape)
df = pd.merge(merge2, sleep, on='SEQN', how = "inner")
print(df.shape)

# Now we want to determine what explanatory variables we are going to keep.




(5533, 26)
(5533, 71)
(5533, 81)


In [81]:
# First we are going to get rid of any response for the depression indicators that weren't 0-3, in other words, the responses that did not mention whether
# or how often they were feeling this way. We are ignoring the variable (DPQ100) that asks about the difficulty the problems have causec because it is
# not directly related to depression score and uses a different encoding system.

depression_cols = ["DPQ010", "DPQ020", "DPQ030", "DPQ040", "DPQ050", "DPQ060", "DPQ070", "DPQ080", "DPQ090"]


# Using a Lamda expression here so we can get the data we want (between 0 and 3) for each collumn at a time.
df[depression_cols] = df[depression_cols].apply(lambda x: x.where(x.between(0, 3)))

# Creating an aggregated depression score out of 27 estimating the severity of the participant's depression.

df["depression_score"] = df[depression_cols].sum(axis=1)
df


,SEQN,DPQ010,DPQ020,DPQ030,DPQ040,DPQ050,DPQ060,DPQ070,DPQ080,DPQ090,...,SLQ310,SLD012,SLQ320,SLQ330,SLD013,SLQ030,SLQ040,SLQ050,SLQ120,depression_score
0,93705.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,...,b'07:00',8.0,b'23:00',b'07:00',8.0,2.000000e+00,5.397605e-79,2.0,5.397605e-79,4.857845e-78
1,93706.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,...,b'10:00',10.5,b'00:30',b'12:00',11.5,1.000000e+00,5.397605e-79,2.0,1.000000e+00,4.857845e-78
2,93708.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,...,b'06:30',8.0,b'22:30',b'06:30',8.0,9.000000e+00,5.397605e-79,2.0,2.000000e+00,4.857845e-78
3,93709.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,b'05:30',7.0,b'22:30',b'05:00',6.5,1.000000e+00,5.397605e-79,2.0,1.000000e+00,0.000000e+00
4,93711.0,1.000000e+00,5.397605e-79,1.000000e+00,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,...,b'05:00',7.0,b'23:00',b'08:00',9.0,2.000000e+00,1.000000e+00,1.0,3.000000e+00,2.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5528,102949.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,...,b'06:00',6.5,b'23:30',b'06:00',6.5,2.000000e+00,5.397605e-79,2.0,2.000000e+00,4.857845e-78
5529,102952.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,...,b'07:00',8.5,b'22:30',b'07:00',8.5,5.397605e-79,5.397605e-79,2.0,5.397605e-79,4.857845e-78
5530,102953.0,1.000000e+00,1.000000e+00,5.397605e-79,1.000000e+00,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,...,b'04:00',6.0,b'23:00',b'04:00',5.0,5.397605e-79,1.000000e+00,1.0,2.000000e+00,3.000000e+00
5531,102954.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,...,b'06:00',8.0,b'00:00',b'07:00',7.0,1.000000e+00,5.397605e-79,2.0,1.000000e+00,4.857845e-78


In [82]:
import numpy as np

# After we created the depression sscore, we need to reformat the data so that it is no longer in scientific notation (a quirk of the original .xpt file)

# Operating only on only the numerical columns to clean the data
num_cols = df.select_dtypes(include = [np.number]).columns
df[num_cols] = df[num_cols].mask(df[num_cols].abs() < 1e-10)

# Some of the data were in byte format so, I used the decode comment if it was
df = df.map(lambda x: x.decode() if isinstance(x, bytes) else x)
df.describe()
df

,SEQN,DPQ010,DPQ020,DPQ030,DPQ040,DPQ050,DPQ060,DPQ070,DPQ080,DPQ090,...,SLQ310,SLD012,SLQ320,SLQ330,SLD013,SLQ030,SLQ040,SLQ050,SLQ120,depression_score
0,93705.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,07:00,8.0,23:00,07:00,8.0,2.0,NaN,2.0,NaN,NaN
1,93706.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,10:00,10.5,00:30,12:00,11.5,1.0,NaN,2.0,1.0,NaN
2,93708.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,06:30,8.0,22:30,06:30,8.0,9.0,NaN,2.0,2.0,NaN
3,93709.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,05:30,7.0,22:30,05:00,6.5,1.0,NaN,2.0,1.0,NaN
4,93711.0,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,05:00,7.0,23:00,08:00,9.0,2.0,1.0,1.0,3.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5528,102949.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,06:00,6.5,23:30,06:00,6.5,2.0,NaN,2.0,2.0,NaN
5529,102952.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,07:00,8.5,22:30,07:00,8.5,NaN,NaN,2.0,NaN,NaN
5530,102953.0,1.0,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,...,04:00,6.0,23:00,04:00,5.0,NaN,1.0,1.0,2.0,3.0
5531,102954.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,06:00,8.0,00:00,07:00,7.0,1.0,NaN,2.0,1.0,NaN


In [83]:
# Now we want to drop all of the observations that don't have a depression score which we do by "masking" our data frame with all of the observations in the depression
# score collumn that don't have NaN

mask = df["depression_score"].notna()
df = df[mask]

# Now we want to drop all of the observations of participants who did not answer at least seven of the questions related to depression screening. We first do this by counting how many
# missing responses we have for each participant then using a similar masking strategy. Originally we wanted the participant to answer all the questons, but this got rid of too mnay observations.

count = df[depression_cols].notna().sum(axis=1)
count_mask = count >= 6
df = df[count_mask]
df



,SEQN,DPQ010,DPQ020,DPQ030,DPQ040,DPQ050,DPQ060,DPQ070,DPQ080,DPQ090,...,SLQ310,SLD012,SLQ320,SLQ330,SLD013,SLQ030,SLQ040,SLQ050,SLQ120,depression_score
6,93713.0,1.0,1.0,2.0,NaN,1.0,1.0,2.0,NaN,NaN,...,04:00,5.5,23:00,06:00,7.0,NaN,NaN,1.0,2.0,8.0
33,93758.0,1.0,1.0,3.0,2.0,1.0,NaN,2.0,1.0,NaN,...,07:30,5.5,02:00,07:30,5.5,NaN,1.0,1.0,4.0,11.0
35,93760.0,3.0,3.0,3.0,3.0,3.0,1.0,NaN,3.0,NaN,...,07:00,8.0,23:00,07:00,8.0,9.0,NaN,2.0,1.0,19.0
53,93782.0,1.0,1.0,2.0,3.0,3.0,NaN,1.0,NaN,NaN,...,06:00,8.0,00:00,07:00,7.0,NaN,NaN,2.0,4.0,11.0
74,93815.0,2.0,3.0,3.0,3.0,1.0,2.0,3.0,NaN,NaN,...,06:00,8.5,22:30,06:00,7.5,1.0,1.0,2.0,2.0,17.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5487,102878.0,1.0,2.0,1.0,2.0,NaN,NaN,3.0,1.0,1.0,...,02:30,8.5,18:00,02:30,8.5,2.0,1.0,1.0,4.0,11.0
5495,102889.0,2.0,2.0,2.0,2.0,1.0,2.0,NaN,NaN,NaN,...,08:00,7.0,02:00,09:00,7.0,3.0,3.0,1.0,4.0,11.0
5508,102911.0,3.0,2.0,3.0,3.0,3.0,1.0,NaN,1.0,NaN,...,04:30,7.0,21:30,04:30,7.0,1.0,1.0,1.0,2.0,16.0
5522,102935.0,2.0,1.0,3.0,3.0,NaN,1.0,3.0,1.0,NaN,...,09:00,8.0,00:00,10:00,10.0,NaN,NaN,1.0,4.0,14.0
